In [1]:
import re
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.phrases import Phrases, Phraser
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [2]:
## load original imdb data (https://ai.stanford.edu/%7Eamaas/data/sentiment/)

import os

def read_imdb_data(data_dir):
    texts = []
    labels = []

    for split in ["train", "test"]:
        for label_name, label in [("pos", 1), ("neg", 0)]:
            folder = os.path.join(data_dir, split, label_name)
            for filename in os.listdir(folder):
                if filename.endswith(".txt"):
                    with open(os.path.join(folder, filename), "r", encoding="utf-8") as f:
                        texts.append(f.read())
                        labels.append(label)
    return texts, np.array(labels)

data_dir = "/Users/shailendra/Downloads/aclImdb"
texts, labels = read_imdb_data(data_dir)

print(f"Loaded {len(texts)} reviews, {sum(labels)} positive, {len(labels)-sum(labels)} negative")

Loaded 50000 reviews, 25000 positive, 25000 negative


In [3]:
# basic preprocessor

def preprocess_google_news(text):
    text = re.sub(r"[^\w\s_]", "", text)
    return text.split()

In [4]:
## tokenize and phrase detection

tokenized_texts = [preprocess_google_news(t) for t in texts]

phrases = Phrases(tokenized_texts, min_count=5, threshold=10)
bigram = Phraser(phrases)

tokenized_texts = [bigram[sent] for sent in tokenized_texts]

In [5]:
## download embeddings
w2v_pretrained = api.load('word2vec-google-news-300')

In [6]:
## MODEL OFF-THE-SHELF WORD2VEC

In [7]:
# Compute OOV + Coverage

total_tokens = 0
oov_tokens = 0

for sentence in tokenized_texts:
    for word in sentence:
        total_tokens += 1
        if word not in w2v_pretrained:
            oov_tokens += 1

oov_rate_pre = oov_tokens / total_tokens
coverage_pre = 1 - oov_rate_pre

print("\nFROZEN EMBEDDINGS")
print("Total tokens:", total_tokens)
print("OOV tokens:", oov_tokens)
print(f"OOV rate: {oov_rate_pre:.2%}")
print(f"Coverage: {coverage_pre:.2%}")


FROZEN EMBEDDINGS
Total tokens: 10697674
OOV tokens: 2097699
OOV rate: 19.61%
Coverage: 80.39%


In [8]:
# build vocab

vocab = {}
for sentence in tokenized_texts:
    for word in sentence:
        if word in w2v_pretrained and word not in vocab:
            vocab[word] = len(vocab) + 1

In [9]:
# convert to sequence 

MAX_LEN = 200

X = [
    [vocab[w] for w in sentence if w in vocab]
    for sentence in tokenized_texts
]

X = pad_sequences(X, maxlen=MAX_LEN, padding="post")

In [10]:
# train and test split

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

In [11]:
# build embedding matrix (from off the shelf word2vec)

embedding_dim = 300
embedding_matrix = np.zeros((len(vocab) + 1, embedding_dim))

for word, i in vocab.items():
    embedding_matrix[i] = w2v_pretrained[word]

In [12]:
model = Sequential([
    Embedding(
        input_dim=len(vocab) + 1,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),
    LSTM(128),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/DL/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │    33,784,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,784,500 (128.88 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 33,784,500 (128.88 MB)

In [13]:
# train model
history = model.fit(
    X_train,
    y_train,
    batch_size=64,
    epochs=5,
    validation_data=(X_test, y_test)
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 87s 139ms/step - accuracy: 0.5411 - loss: 0.6863 - val_accuracy: 0.5270 - val_loss: 0.6864
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 135ms/step - accuracy: 0.5338 - loss: 0.6889 - val_accuracy: 0.5234 - val_loss: 0.6864
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 85s 136ms/step - accuracy: 0.6622 - loss: 0.5896 - val_accuracy: 0.8269 - val_loss: 0.3909
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 134ms/step - accuracy: 0.8476 - loss: 0.3568 - val_accuracy: 0.8622 - val_loss: 0.3236
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 135ms/step - accuracy: 0.8630 - loss: 0.3251 - val_accuracy: 0.8708 - val_loss: 0.3124


In [14]:
# model 1 accuracy

loss, acc = model.evaluate(X_test, y_test)

print("\nRESULTS")
print("Frozen Accuracy:", acc)

print("\nCoverage")
print(f"Frozen Coverage: {coverage_pre:.2%}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 40ms/step - accuracy: 0.8708 - loss: 0.3124

RESULTS
Frozen Accuracy: 0.8708000183105469

Coverage
Frozen Coverage: 80.39%
